# Análisis bivariado

In [1]:
from eda_utils import *   # carga y prepara la base (capítulo 1)

## Precio y área

In [2]:
m = viv[viv.estrato.isin(ESTRATOS)].sample(6000, random_state=1)
fig = px.scatter(m, x="area", y="precio_M", color="estrato_txt", log_x=True, log_y=True, opacity=0.6,
                 category_orders={"estrato_txt": NOMBRES_ESTRATO}, color_discrete_sequence=COLORES_ESTRATO,
                 custom_data=["barrio_txt", "precio_txt"],
                 labels={"area": "área (m², escala log)", "precio_M": "precio (millones de 2025, escala log)",
                         "estrato_txt": ""})
fig.update_traces(marker_size=5, hovertemplate="<b>%{customdata[0]}</b><br>%{x:.0f} m²<br>%{customdata[1]}<extra></extra>")
mostrar(fig, "Precio y área (6.000 ventas al azar; clic en la leyenda para filtrar)", alto=520)

rho, p = stats.spearmanr(viv.precio_real, viv.area)
z, ee = np.arctanh(rho), 1 / np.sqrt(len(viv) - 3)
li, ls = np.tanh([z - 1.96 * ee, z + 1.96 * ee])
anotar("¿El precio sube con el área?", "Correlación de Spearman", rho, p,
       f"rho = {rho:.2f} (IC 95%: {li:.2f} a {ls:.2f})")

Correlación de Spearman: estadístico = 0.481   p = < 0,001   rho = 0.48 (IC 95%: 0.47 a 0.49)


A más área, más precio (Spearman 0,48), pero para un mismo tamaño el precio cambia
mucho según el estrato: los puntos rojos (estrato 6) quedan arriba de los azules en
todo el rango.

## Estrato

In [3]:
d = viv[viv.estrato.isin(ESTRATOS)]
fig = px.box(d, x="estrato_txt", y="pm2_M", color="estrato_txt", points=False,
             category_orders={"estrato_txt": NOMBRES_ESTRATO}, color_discrete_sequence=COLORES_ESTRATO,
             labels={"estrato_txt": "", "pm2_M": "millones por m² (pesos de 2025)"})
fig.update_yaxes(range=[0, d.pm2_M.quantile(0.99)])
fig.update_layout(showlegend=False)
mostrar(fig, "Precio por m² según estrato")

(d.groupby("estrato_txt")
   .agg(ventas=("precio", "size"), precio_mediano=("precio_real", "median"),
        m2_mediano=("precio_m2_real", "median"), area_mediana=("area", "median"), edad_mediana=("edad", "median"))
   .assign(precio_mediano=lambda t: t.precio_mediano.map(cop), m2_mediano=lambda t: t.m2_mediano.map(cop)))

,ventas,precio_mediano,m2_mediano,area_mediana,edad_mediana
estrato_txt,,,,,
Estrato 1,3062,$127.1 M,$2.4 M,48.00,6.00
Estrato 2,2188,$120.8 M,$1.4 M,72.50,23.00
Estrato 3,3573,$179.7 M,$2.7 M,75.00,11.00
Estrato 4,3910,$319.0 M,$4.6 M,73.00,6.00
Estrato 5,806,$381.7 M,$4.3 M,92.00,8.00
Estrato 6,894,$812.8 M,$5.3 M,143.00,6.00


In [4]:
H, p, eps2 = kruskal_por_grupo(viv, "precio_m2_real", "estrato", ESTRATOS)
anotar("¿El precio por m² difiere entre estratos?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")

x4 = viv.loc[viv.estrato == "Medio_4", "precio_m2_real"]
x5 = viv.loc[viv.estrato == "Medio_Alto_5", "precio_m2_real"]
U, p = stats.mannwhitneyu(x4, x5)
anotar("¿El estrato 5 es más caro por m² que el 4?", "Mann-Whitney", U, p,
       f"medianas {cop(x4.median())} y {cop(x5.median())}")

Kruskal-Wallis: estadístico = 5,177.546   p = < 0,001   épsilon² = 0.36
Mann-Whitney: estadístico = 1,634,590.500   p = 0,094   medianas $4.6 M y $4.3 M


El estrato explica el 36% de la variación del precio por m² (Kruskal-Wallis,
épsilon² = 0,36). Hay dos excepciones. El estrato 1 es más caro por m² que el 2 porque
son apartamentos nuevos y pequeños, frente a casas de 23 años. Y entre los estratos 4 y
5 no hay diferencia significativa (p = 0,09).

## Barrio y localidad

Usamos solo las ventas con el área medida en la unidad, porque el área tomada del
edificio distorsiona el precio por m².

In [5]:
medidas = viv[viv.origen_caract == "unidad"]   # área medida en la unidad, no tomada del edificio
por_barrio = (medidas[medidas.barrio.notna() & ~medidas.barrio.isin(NO_BARRIOS)]
              .groupby("barrio").agg(n=("precio", "size"), pm2=("precio_m2_real", "median"),
                                     estrato=("estrato_num", "median"))
              .query("n >= 40").sort_values("pm2"))

fig = make_subplots(1, 2, subplot_titles=("15 más baratos", "15 más caros"), horizontal_spacing=0.28)
for i, t in enumerate([por_barrio.head(15), por_barrio.tail(15)], 1):
    fig.add_bar(x=t.pm2 / 1e6, y=t.index.str.title(), orientation="h", row=1, col=i,
                marker_color=AZUL if i == 1 else ROJO, customdata=np.stack([t.n, t.estrato], axis=1),
                hovertemplate="<b>%{y}</b><br>%{x:.2f} M por m²<br>%{customdata[0]} ventas<br>"
                              "estrato mediano %{customdata[1]}<extra></extra>")
fig.update_xaxes(title_text="millones por m² (pesos de 2025)")
fig.update_layout(showlegend=False)
mostrar(fig, f"Precio mediano por m² en los barrios con 40 ventas o más ({len(por_barrio)} barrios)", alto=520)

print(f"Diferencia entre el más caro y el más barato: {por_barrio.pm2.max() / por_barrio.pm2.min():.1f} veces")

H, p, eps2 = kruskal_por_grupo(medidas, "precio_m2_real", "barrio", por_barrio.index)
anotar("¿El precio por m² difiere entre barrios?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")
localidades = [l for l in viv.localidad.dropna().unique() if l != "RURAL"]
H, p, eps2 = kruskal_por_grupo(viv, "precio_m2_real", "localidad", localidades)
anotar("¿El precio por m² difiere entre localidades?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")

Diferencia entre el más caro y el más barato: 7.9 veces
Kruskal-Wallis: estadístico = 5,354.086   p = < 0,001   épsilon² = 0.58
Kruskal-Wallis: estadístico = 4,770.237   p = < 0,001   épsilon² = 0.46


Palmas del Río cuesta 7,9 veces por m² lo que Santo Domingo de Guzmán. El barrio explica
más (épsilon² = 0,58) que la localidad (0,46) y que el estrato (0,36), así que el modelo
necesita ubicación fina, no solo el estrato.

## Evolución en el tiempo

In [6]:
serie = viv.groupby("anio").agg(n=("precio", "size"), nominal=("precio_m2", "median"),
                                 real=("precio_m2_real", "median"))
lineas = serie[serie.n >= 100]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=serie.index, y=serie.n, name="ventas", marker_color="#DCE4EC", secondary_y=False,
            hovertemplate="%{x}: %{y} ventas<extra></extra>")
fig.add_scatter(x=lineas.index, y=lineas.nominal / 1e6, name="en pesos de cada año", mode="lines+markers",
                line=dict(color=GRIS, width=2, dash="dot"), hovertemplate="%{x}: %{y:.2f} M<extra></extra>",
                secondary_y=True)
fig.add_scatter(x=lineas.index, y=lineas.real / 1e6, name="en pesos de 2025", mode="lines+markers",
                line=dict(color=ROJO, width=3), hovertemplate="%{x}: %{y:.2f} M<extra></extra>",
                secondary_y=True)
# las barras (ventas) van en el eje derecho y detrás de las líneas
fig.update_yaxes(title_text="ventas", secondary_y=False, showgrid=False, side="right")
fig.update_yaxes(title_text="millones por m² (mediana)", secondary_y=True, side="left",
                 tickmode="auto", tickformat=".1f", showgrid=True)
fig.update_xaxes(dtick=1)
fig.update_layout(legend=dict(orientation="h", y=1.08))
mostrar(fig, "Precio mediano por m² por año, con y sin inflación")

recientes = viv[viv.anio >= 2019]
H, p, eps2 = kruskal_por_grupo(recientes, "precio_m2_real", "anio", range(2019, 2026))
anotar("¿El m² real cambia entre 2019 y 2025?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.3f}")

serie.assign(nominal=serie.nominal.map(cop), real=serie.real.map(cop))

Kruskal-Wallis: estadístico = 682.969   p = < 0,001   épsilon² = 0.045


,n,nominal,real
anio,,,
2016,1,$427 mil,$718 mil
2017,116,$2.3 M,$3.6 M
2018,295,$2.6 M,$4.1 M
2019,2469,$2.0 M,$3.0 M
2020,1161,$2.1 M,$3.1 M
2021,1832,$1.7 M,$2.4 M
2022,2421,$1.8 M,$2.3 M
2023,2951,$2.3 M,$2.7 M
2024,1335,$3.2 M,$3.5 M


En pesos corrientes el m² parecía subir 80% entre 2022 y 2025. En pesos de 2025 la
mediana baja de 3,0 M en 2019 a 2,3 M en 2022 y vuelve a 3,3 M en 2025. Parte de esos
movimientos es cambio en qué se vendió cada año, y eso se corrige en la comparación
siguiente.

### Comparación con el índice oficial

La mediana por año mezcla dos cosas: el cambio de precio y el cambio en qué se vendió
cada año (más o menos Alameda del Río, más o menos estrato 6). Para separarlas
calculamos un índice ajustado por composición: una regresión del precio por m² real con
el año, controlando barrio, estrato, área y tipo de predio. Lo comparamos con el Índice
de Precios de la Vivienda Usada (IPVU) real del Banco de la República. Usamos la serie
nacional, que incluye a Barranquilla: el Banco no publica el IPVU por ciudad. Los datos están en `datos/ipvu_nacional_banrep.csv`.

In [7]:
import statsmodels.formula.api as smf

# Índice ajustado por composición: regresión del log del m² real con efectos de año,
# controlando barrio, estrato, área y tipo de predio. El efecto de cada año es el cambio
# de precio para una vivienda "igual" en todo lo demás.
d = viv[viv.anio.between(2019, 2025)].copy()
d["barrio_m"] = d.barrio.fillna("SIN BARRIO").astype(str)
d["tipo"] = d.condicion_predio.fillna("SIN DATO").astype(str)
d["estrato_m"] = d.estrato_txt.astype(str)
d["log_pm2"] = np.log(d.precio_m2_real)
d["log_area"] = np.log(d.area)
hedonico = smf.ols("log_pm2 ~ C(anio) + C(barrio_m) + C(estrato_m) + log_area + C(tipo)", data=d).fit()
ic = hedonico.conf_int()
anios = range(2019, 2026)
indice = pd.Series({a: 100.0 if a == 2019 else 100 * np.exp(hedonico.params[f"C(anio)[T.{a}]"]) for a in anios})
inf = pd.Series({a: 100.0 if a == 2019 else 100 * np.exp(ic.loc[f"C(anio)[T.{a}]", 0]) for a in anios})
sup = pd.Series({a: 100.0 if a == 2019 else 100 * np.exp(ic.loc[f"C(anio)[T.{a}]", 1]) for a in anios})

# IPVU real nacional del Banco de la República (promedio de los cuatro trimestres)
ipvu = pd.read_csv("datos/ipvu_nacional_banrep.csv", parse_dates=["fecha"])
ipvu_anual = ipvu.groupby(ipvu.fecha.dt.year).ipvu_real.mean().loc[2019:2025]
ipvu_indice = 100 * ipvu_anual / ipvu_anual.loc[2019]
simple = serie.real.loc[2019:2025]
simple_indice = 100 * simple / simple.loc[2019]

fig = go.Figure()
fig.add_scatter(x=list(anios) + list(anios)[::-1], y=list(sup) + list(inf)[::-1], fill="toself",
                fillcolor="rgba(199,82,42,0.15)", line=dict(width=0), hoverinfo="skip", name="IC 95% del ajustado")
fig.add_scatter(x=simple_indice.index, y=simple_indice, name="mediana simple (esta base)", mode="lines+markers",
                line=dict(color=GRIS, width=2, dash="dot"), hovertemplate="%{x}: %{y:.0f}<extra></extra>")
fig.add_scatter(x=indice.index, y=indice, name="ajustado por composición (esta base)", mode="lines+markers",
                line=dict(color=ROJO, width=3), hovertemplate="%{x}: %{y:.0f}<extra></extra>")
fig.add_scatter(x=ipvu_indice.index, y=ipvu_indice, name="IPVU real nacional (Banco de la República)",
                mode="lines+markers", line=dict(color=AZUL, width=3), hovertemplate="%{x}: %{y:.0f}<extra></extra>")
fig.add_hline(y=100, line_dash="dash", line_color="#999")
fig.update_yaxes(title="índice, 2019 = 100 (precios reales)")
fig.update_xaxes(dtick=1)
fig.update_layout(legend=dict(orientation="h", y=1.12))
mostrar(fig, "Precio real de la vivienda: nuestra base frente al índice oficial", alto=500)

print(f"R² del modelo de ajuste: {hedonico.rsquared:.2f}   (n = {int(hedonico.nobs):,})")
pd.DataFrame({"mediana simple": simple_indice.round(1), "ajustado por composición": indice.round(1),
              "IPVU real nacional": ipvu_indice.round(1)})

R² del modelo de ajuste: 0.61   (n = 15,047)


,mediana simple,ajustado por composición,IPVU real nacional
2019,100.00,100.00,100.00
2020,101.60,107.50,99.70
2021,78.90,98.10,101.70
2022,77.00,92.40,98.80
2023,87.90,90.50,94.40
2024,115.30,108.80,95.80
2025,110.80,97.70,98.80


La mediana simple exagera los movimientos: cae 23% en 2022 y sube 15% en 2024, pero
buena parte es cambio en qué se vendió. Con el ajuste por composición (R² = 0,61) la
serie es mucho más estable y termina 2025 en 97,7, muy cerca del IPVU real nacional
(98,8). Las dos fuentes coinciden en lo principal: descontando la inflación, el precio
de la vivienda en 2025 está prácticamente igual que en 2019. La única diferencia grande
es 2024 (108,8 contra 95,8), el año con menos ventas en nuestra base.

Esto valida la base con una fuente oficial independiente y corrige la lectura de la
mediana simple.

## Características de la vivienda

In [8]:
fig = make_subplots(1, 2, subplot_titles=("Precio según número de baños", "Precio por m² según piso"))
for b in range(1, 6):
    fig.add_box(y=viv.loc[viv.banios == b, "precio_M"], name=str(b), boxpoints=False,
                marker_color=AZUL, row=1, col=1)
g = viv[viv.piso.between(1, 15)].groupby(viv.piso.round()).pm2_M.median()
fig.add_scatter(x=g.index, y=g.values, mode="lines+markers", line=dict(color=ROJO, width=3), row=1, col=2,
                hovertemplate="piso %{x}: %{y:.2f} M por m²<extra></extra>")
fig.update_yaxes(title_text="millones de 2025", range=[0, viv.precio_M.quantile(0.97)], row=1, col=1)
fig.update_yaxes(title_text="millones por m²", row=1, col=2)
fig.update_xaxes(title_text="baños", row=1, col=1)
fig.update_xaxes(title_text="piso", row=1, col=2)
fig.update_layout(showlegend=False)
mostrar(fig, "Baños y piso frente al precio")

filas = []
for col in ["banios", "habitaciones", "plantas", "piso", "edad"]:
    d = viv[[col, "precio_real", "precio_m2_real"]].dropna()
    r1, p1 = stats.spearmanr(d[col], d.precio_real)
    r2, p2 = stats.spearmanr(d[col], d.precio_m2_real)
    filas.append({"variable": col, "rho con precio": round(r1, 2), "p": p_texto(p1),
                  "rho con precio/m²": round(r2, 2), "p ": p_texto(p2)})
pd.DataFrame(filas).set_index("variable")

,rho con precio,p,rho con precio/m²,p
variable,,,,
banios,0.55,"< 0,001",0.34,"< 0,001"
habitaciones,0.30,"< 0,001",-0.10,"< 0,001"
plantas,0.02,"0,025",-0.24,"< 0,001"
piso,0.37,"< 0,001",0.64,"< 0,001"
edad,-0.14,"< 0,001",-0.32,"< 0,001"


Los baños son la característica que más se relaciona con el precio (0,55). El piso se
relaciona más con el precio por m² (0,64) que con el precio total (0,37): los pisos altos
son apartamentos de zonas caras. Las viviendas más viejas son algo más baratas por m²
(−0,32). El número de plantas no importa.

## Correlaciones

In [9]:
numericas = {"log_precio": "log precio", "area": "área", "precio_m2_real": "precio/m²", "avaluo": "avalúo",
             "estrato_num": "estrato", "banios": "baños", "habitaciones": "habitaciones", "piso": "piso",
             "edad": "edad", "lat": "latitud", "lon": "longitud", "anio": "año"}
corr = viv[list(numericas)].rename(columns=numericas).corr(method="spearman").round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
fig.update_traces(textfont_size=11)
mostrar(fig, "Correlación de Spearman entre variables numéricas", alto=620)

Con el precio se relacionan sobre todo el estrato (0,71), el precio por m² (0,70), el
avalúo (0,64) y la latitud (0,58); después vienen baños y área. Estrato, latitud y avalúo
dicen casi lo mismo sobre la ubicación, lo que genera colinealidad en un modelo lineal.